# 40.12 Межканальное влияние реокардиомонитора РНЦХ

В тесте кабели отключались от прибора без отклеивания электродов. Поэтому
изменение канала 1 при смене состояния соседнего канала — экспериментальный
факт для этой ревизии. Ноутбук количественно описывает размер эффекта, не
приписывая ему электрический или физиологический механизм.


In [ ]:
import json
import os
from pathlib import Path

import numpy as np

phase = np.linspace(0.0, 1.0, 101)
a = np.sin(2 * np.pi * phase)
b = 1.2 * a + 0.1
assert np.ptp(b - b.mean()) / np.ptp(a - a.mean()) > 1.19
REAL_MODE = os.environ.get("KALMYKOV_RUN_REAL", "0") == "1"
print("40.12 synthetic_self_test: passed")


In [ ]:
if not REAL_MODE:
    print("40.12 real_data_status: blocked_until_accepted_record_and_channel_state_manifests")
else:
    import hashlib
    import pandas as pd

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    config_path = Path(os.environ["KALMYKOV_EXP03_CONFIG"]).expanduser().resolve()
    config = json.loads(config_path.read_text(encoding="utf-8"))
    record_spec = config.get("record_manifest", {})
    switch_spec = config.get("channel_switch_analysis", {})
    if record_spec.get("status") != "accepted" or switch_spec.get("annotation_status") != "accepted":
        raise RuntimeError("Не приняты QC-манифест записей и разметка состояний каналов")
    record_manifest_path = Path(record_spec["path"]).expanduser().resolve()
    record_manifest = json.loads(record_manifest_path.read_text(encoding="utf-8"))
    if record_manifest.get("status") != "accepted" or record_manifest.get("experiment_id") != "exp03":
        raise RuntimeError("Неподходящий QC-манифест")
    by_id = {item["record_id"]: item for item in record_manifest["records"] if item.get("include") is True}
    candidates = [item for item in config["recordings"] if item.get("role") == "channel_switch_test"]
    if len(candidates) != 1 or candidates[0]["record_id"] not in by_id:
        raise RuntimeError("Должна быть ровно одна принятая запись channel_switch_test")
    record = by_id[candidates[0]["record_id"]]
    sidecar_path = Path(switch_spec["state_sidecar"]).expanduser().resolve()
    sidecar = json.loads(sidecar_path.read_text(encoding="utf-8"))
    if sidecar.get("status") != "accepted" or sidecar.get("record_id") != record["record_id"] or sidecar.get("input_sha256") != record["input_sha256"]:
        raise RuntimeError("Разметка состояний не принята или относится к другой записи")
    data_root = Path(config["data_root"]).expanduser().resolve()
    csv_root = (data_root / config["csv_subdir"]).resolve()
    source = (data_root / record["relative_path"]).resolve()
    source.relative_to(csv_root)
    if sha256_file(source) != record["input_sha256"]:
        raise RuntimeError("CSV изменился после QC")
    frame = pd.read_csv(source)
    if list(frame.columns) != config["source_columns"]:
        raise RuntimeError("Схема CSV не совпала")
    frame.columns = ["time_s", "rheo_1_mohm", "base_1_ohm", "qs_1_ohm", "ecg_v", "rheo_2_mohm", "base_2_ohm", "qs_2_ohm"]
    time_s = frame["time_s"].to_numpy(float)
    phase_grid = np.linspace(0.0, 1.0, 201)
    grouped = {(1,): [], (1, 2): []}
    for segment in sidecar.get("segments", []):
        state = tuple(segment.get("actual_active_channels", []))
        if state not in grouped:
            continue
        left, right = float(segment["start_s"]), float(segment["stop_s"])
        mask = (time_s >= left) & (time_s <= right)
        if mask.sum() < 10:
            raise RuntimeError("Сегмент переключения слишком короток")
        segment_time = time_s[mask]
        phase_local = (segment_time - segment_time[0]) / (segment_time[-1] - segment_time[0])
        rheo = frame.loc[mask, "rheo_1_mohm"].to_numpy(float)
        base = frame.loc[mask, "base_1_ohm"].to_numpy(float)
        grouped[state].append({
            "baseline_ohm": float(np.median(base)),
            "waveform_mohm": np.interp(phase_grid, phase_local, rheo - np.mean(rheo)),
        })
    if any(len(grouped[state]) == 0 for state in grouped):
        raise RuntimeError("Нужны сегменты как для канала 1 отдельно, так и для каналов 1+2")
    summaries = {}
    for state, rows in grouped.items():
        waves = np.asarray([row["waveform_mohm"] for row in rows])
        baselines = np.asarray([row["baseline_ohm"] for row in rows])
        summaries[state] = {
            "n_segments": len(rows), "baseline_mean_ohm": float(baselines.mean()),
            "baseline_sd_ohm": float(baselines.std(ddof=1)) if len(rows) > 1 else None,
            "mean_signed_waveform_mohm": waves.mean(axis=0),
            "waveform_ptp_mohm": float(np.ptp(waves.mean(axis=0))),
        }
    one, both = summaries[(1,)], summaries[(1, 2)]
    if one["baseline_mean_ohm"] == 0 or one["waveform_ptp_mohm"] == 0:
        raise RuntimeError("Нулевой знаменатель метрики")
    difference = both["mean_signed_waveform_mohm"] - one["mean_signed_waveform_mohm"]
    l2_denominator = float(np.linalg.norm(one["mean_signed_waveform_mohm"]))
    metrics = {
        "baseline_ratio_both_to_channel1_only": both["baseline_mean_ohm"] / one["baseline_mean_ohm"],
        "signed_waveform_difference_mohm": difference.tolist(),
        "peak_to_peak_ratio_both_to_channel1_only": both["waveform_ptp_mohm"] / one["waveform_ptp_mohm"],
        "relative_l2_difference": None if l2_denominator == 0 else float(np.linalg.norm(difference) / l2_denominator),
        "repeatability_status": "estimated_from_repeated_segments" if min(one["n_segments"], both["n_segments"]) >= 2 else "not_estimated_single_segment",
    }
    artifact = {
        "schema_version": 2, "analysis": "40.12_interchannel_effect",
        "status": "quantified_observed_effect_mechanism_unknown", "record_id": record["record_id"],
        "input_sha256": record["input_sha256"], "phase_grid": phase_grid.tolist(),
        "states": {
            "channel_1_only": {**one, "mean_signed_waveform_mohm": one["mean_signed_waveform_mohm"].tolist()},
            "channels_1_and_2": {**both, "mean_signed_waveform_mohm": both["mean_signed_waveform_mohm"].tolist()},
        },
        "metrics": metrics, "cause": "not_investigated",
    }
    out_dir = Path(config["derived_root"]).expanduser().resolve() / "exp03" / "analysis"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "40.12_interchannel_effect.json"
    out_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print("40.12 real_data_status: artifact_written", out_path)


## Интерпретация

Метрики показывают, насколько меняется наблюдаемый канал 1 при подключении
соседнего канала. Они не доказывают шунтирование, перераспределение тока или
изменение чувствительности именно к сердцу.
